# LEFT JOIN and finding rows with no match

Practice notebook for the datafromzero SQL lesson **LEFT JOIN and finding rows with no match** (S20). The engine is DuckDB and the data is
**cafe-sales**: five tables from a fictional chain of three cafes (synthetic data, CC BY 4.0).

Run the cells from top to bottom. The worked example is written out so you can run it and read the results. The
exercises start empty or half done: write your own query there. The solutions are in the lesson, not here.

Each query sits between `con.sql("""` and `""").show()`: only the text inside the triple quotes is SQL, and `.show()`
prints the result as a table. The cells leave out the closing `;` that the lesson shows.

## Setup

Run this cell first. It builds an empty database in memory and loads the five cafe-sales tables into it.

In [ ]:
%pip install -q duckdb==2.0.0  # the version these lessons were checked with
import duckdb, urllib.request

URL = "https://raw.githubusercontent.com/datafromzero/datafromzero.github.io/v0.1.0/datasets/cafe-sales/v1/cafe-sales.sql"
script = urllib.request.urlopen(URL).read().decode("utf-8")

con = duckdb.connect()  # an empty database that lives in memory
con.execute(script)     # creates and fills the five cafe-sales tables
print("cafe-sales is loaded")

## Worked example

### Step 1. Count what an inner join keeps

In [ ]:
con.sql("""
SELECT COUNT(*) AS joined_rows,
       COUNT(DISTINCT c.customer_id) AS customers_paired
FROM customers AS c
INNER JOIN orders AS o ON o.customer_id = c.customer_id
""").show()

### Step 2. Keep everyone with LEFT JOIN

In [ ]:
con.sql("""
SELECT COUNT(*) AS joined_rows,
       COUNT(o.order_id) AS paired_rows,
       COUNT(DISTINCT c.customer_id) AS customers
FROM customers AS c
LEFT JOIN orders AS o ON o.customer_id = c.customer_id
""").show()

### Step 3. See the NULLs

In [ ]:
con.sql("""
SELECT c.customer_id, c.loyalty_tier, o.order_id, o.status
FROM customers AS c
LEFT JOIN orders AS o ON o.customer_id = c.customer_id
WHERE c.customer_id IN (42, 140, 141, 147)
ORDER BY c.customer_id, o.order_id
""").show()

### Step 4. Keep only the unpaired rows: the anti-join

In [ ]:
con.sql("""
SELECT c.customer_id, c.signup_date, c.loyalty_tier
FROM customers AS c
LEFT JOIN orders AS o ON o.customer_id = c.customer_id
WHERE o.order_id IS NULL
ORDER BY c.customer_id
""").show()

### Step 5. Verify with a second route

In [ ]:
con.sql("""
SELECT c.customer_id
FROM customers AS c
ANTI JOIN orders AS o ON o.customer_id = c.customer_id
ORDER BY c.customer_id
""").show()

### Step 6. The tempting way to ask about completed orders

In [ ]:
con.sql("""
SELECT COUNT(*) AS joined_rows,
       COUNT(DISTINCT c.customer_id) AS customers
FROM customers AS c
LEFT JOIN orders AS o ON o.customer_id = c.customer_id
WHERE o.status = 'completed'
""").show()

### Step 7. Put the condition in `ON`

In [ ]:
con.sql("""
SELECT c.customer_id
FROM customers AS c
LEFT JOIN orders AS o
       ON o.customer_id = c.customer_id AND o.status = 'completed'
WHERE o.order_id IS NULL
ORDER BY c.customer_id
""").show()

### Step 8. Check the picture

In [ ]:
con.sql("""
SELECT COUNT(*) AS right_rows
FROM customers AS c
RIGHT JOIN orders AS o ON o.customer_id = c.customer_id
""").show()
con.sql("""
SELECT COUNT(*) AS full_rows
FROM customers AS c
FULL JOIN orders AS o ON o.customer_id = c.customer_id
""").show()

## Exercise 1. Which products have never been sold?

Given: the grain is one product, and the table to keep whole is `products`. Add a `LEFT JOIN` to `order_items` on `product_id`, and keep only the products with no line.

In [ ]:
con.sql("""
SELECT p.product_name
FROM products AS p
-- add the LEFT JOIN and the filter here
""").show()

## Exercise 2. Orders per store in January 2026

How many orders did each store take in January 2026? Every store must be listed. Before you run anything, write one sentence in the cell below: which stores must appear, how many rows you expect, and whether the January condition goes in `ON` or in `WHERE`, and why. Use the date range from the dates lesson: from January 1, up to but not including February 1.

In [ ]:
# My expected result, in words:
#

In [ ]:
# Your query:

## Exercise 3. Spot the flaw

A colleague wants the customers with no completed order and runs this query. Run it, then find out what went wrong and write the right query.

In [ ]:
con.sql("""
SELECT c.customer_id
FROM customers AS c
LEFT JOIN orders AS o ON o.customer_id = c.customer_id
WHERE o.status <> 'completed'
""").show()

In [ ]:
# Your corrected query: